<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания: 16


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

<b> Описание задачи: </b>

Создать базовый класс PaymentMethod в C#, который будет представлять
различные способы оплаты. На основе этого класса разработать 2-3 производных
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из
классов должны быть реализованы новые атрибуты и методы, а также
переопределены некоторые методы базового класса для демонстрации
полиморфизма.

<b> Требования к базовому классу PaymentMethod: </b>

<b> • Атрибуты: </b> ID способа оплаты (PaymentMethodId), Название способа оплаты
(MethodName), Минимальная сумма (MinAmount).

<b> • Методы: </b>

- ProcessPayment(decimal amount): метод для обработки платежа
указанной суммы.

- CheckMinimumAmount(decimal amount): метод для проверки
минимальной суммы платежа.

- GetPaymentDetails(): метод для получения деталей способа оплаты.

<b> Требования к производным классам: </b>

1. ОнлайнОплата (OnlinePayment): Должен содержать дополнительные
атрибуты, такие как URL платежной системы (PaymentUrl).
Метод ProcessPayment() должен быть переопределен для включения URL
платежной системы в процесс оплаты.
2. БанковскийПеревод (BankTransfer): Должен содержать дополнительные
атрибуты, такие как Банковские данные (BankData).
Метод CheckMinimumAmount() должен быть переопределен для проверки
минимальной суммы платежа с учетом банковских комиссий.
3. Наличные (CashPayment) (если требуется третий класс): Должен содержать
дополнительные атрибуты, такие как Место выдачи наличных
(CashPickupPoint). Метод GetPaymentDetails() должен быть переопределен
для отображения места выдачи наличных.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) и реализуйте полиморфизм с перекрытием и прегегрузкой методов, а также generic классы

<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [11]:
using System;
using System.Collections.Generic;
using System.Linq;

public interface ITransactionLogger
{
    void LogTransaction(string transactionDetails);
    string GetTransactionHistory();
}

public interface ISecurityValidator
{
    bool ValidateSecurity();
    void SetSecurityLevel(int level);
}

public interface ICurrencyConverter
{
    decimal ConvertAmount(decimal amount, string targetCurrency);
    string DefaultCurrency { get; set; }
}

public class PaymentCollection<T> where T : PaymentMethod
{
    private List<T> _payments = new List<T>();

    public void AddPayment(T payment)
    {
        _payments.Add(payment);
        Console.WriteLine($"Добавлен: {payment.MethodName}");
    }

    public T FindPayment(Predicate<T> predicate)
    {
        return _payments.Find(predicate);
    }

    public void ProcessAll(decimal amount)
    {
        Console.WriteLine($"Обработка коллекции ({_payments.Count} элементов):");
        foreach (var payment in _payments)
        {
            payment.ProcessPaymentWithCheck(amount);
        }
    }

    public int Count => _payments.Count;
}

public abstract class PaymentMethod : ITransactionLogger, ISecurityValidator
{
    public int PaymentMethodId { get; set; }
    public string MethodName { get; set; }
    public decimal MinAmount { get; set; }
    public DateTime CreatedDate { get; set; }
    public bool IsActive { get; set; }
    public string Currency { get; set; }
    public int SecurityLevel { get; set; }
    protected List<string> TransactionHistory { get; set; }
    
    public string Category { get; set; }
    public int Priority { get; set; }
    public decimal SuccessRate { get; set; }
    private int _todayTransactionCount;
    
    protected PaymentMethod() 
    {
        CreatedDate = DateTime.Now;
        IsActive = true;
        Currency = "RUB";
        SecurityLevel = 1;
        TransactionHistory = new List<string>();
        Category = "Общие";
        Priority = 1;
        SuccessRate = 95.0m;
        _todayTransactionCount = 0;
    }
    
    protected PaymentMethod(int id, string name, decimal minAmount) : this()
    {
        PaymentMethodId = id;
        MethodName = name;
        MinAmount = minAmount;
    }

    public virtual void ProcessPayment(decimal amount)
    {
        _todayTransactionCount++;
        Console.WriteLine($"Транзакция через {MethodName}: {amount} RUB");
        LogTransaction($"Транзакция: {amount}");
        
        if (new Random().Next(100) < SuccessRate)
        {
            Console.WriteLine("Транзакция успешна");
        }
        else
        {
            Console.WriteLine("Транзакция не удалась");
        }
    }

    public virtual bool CheckMinimumAmount(decimal amount)
    {
        if (amount >= MinAmount)
        {
            Console.WriteLine("Средств достаточно");
            return true;
        }
        else
        {
            Console.WriteLine("Недостаточно средств");
            return false;
        }
    }

    public virtual void GetPaymentDetails()
    {
        Console.WriteLine($"ID: {PaymentMethodId}, Способ: {MethodName}, Мин. сумма: {MinAmount} RUB");
    }

    public virtual void ProcessPayment(decimal amount, string reference)
    {
        Console.WriteLine($"Транзакция с референсом: {reference}");
        ProcessPayment(amount);
    }

    public void ProcessPaymentWithCheck(decimal amount)
    {
        if (CheckMinimumAmount(amount) && ValidateSecurity())
        {
            ProcessPayment(amount);
        }
        else
        {
            Console.WriteLine("Транзакция отменена");
        }
    }

    public virtual void UpdatePriority(int newPriority)
    {
        Priority = newPriority;
        Console.WriteLine($"Приоритет изменен на: {newPriority}");
    }

    public virtual string GetStatus()
    {
        return $"{MethodName}: {(IsActive ? "Активен" : "Неактивен")}, Приоритет: {Priority}";
    }

    public void LogTransaction(string transactionDetails)
    {
        string logEntry = $"{DateTime.Now:dd.MM.yyyy HH:mm:ss} - {transactionDetails}";
        TransactionHistory.Add(logEntry);
    }

    public string GetTransactionHistory()
    {
        return TransactionHistory.Count == 0 
            ? "История пуста" 
            : string.Join("\n", TransactionHistory);
    }

    public virtual bool ValidateSecurity()
    {
        return SecurityLevel >= 1 && IsActive;
    }

    public virtual void SetSecurityLevel(int level)
    {
        SecurityLevel = level;
    }
}

class OnlinePayment : PaymentMethod, ICurrencyConverter
{
    public string PaymentUrl { get; set; }
    public string DefaultCurrency { get; set; }
    public string GatewayProvider { get; set; }
    public int TimeoutSeconds { get; set; }
    
    public OnlinePayment() : base() 
    {
        DefaultCurrency = "RUB";
        GatewayProvider = "DefaultGateway";
        TimeoutSeconds = 30;
    }
    
    public OnlinePayment(string paymentUrl) : base(1, "Онлайн", 100m)
    {
        PaymentUrl = paymentUrl;
        DefaultCurrency = "RUB";
    }
    
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Онлайн платеж через {GatewayProvider}");
        base.ProcessPayment(amount);
        Console.WriteLine($"URL: {PaymentUrl}");
    }
    
    public void ProcessPayment(decimal amount, bool useBackupGateway)
    {
        if (useBackupGateway)
        {
            Console.WriteLine("Использование резервного шлюза");
            ProcessPayment(amount);
        }
        else
        {
            ProcessPayment(amount);
        }
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Провайдер: {GatewayProvider}, URL: {PaymentUrl}");
    }
    
    public decimal ConvertAmount(decimal amount, string targetCurrency)
    {
        decimal convertedAmount = amount * 0.85m;
        Console.WriteLine($"Конвертация: {amount} {Currency} → {convertedAmount} {targetCurrency}");
        return convertedAmount;
    }
}

class BankTransfer : PaymentMethod
{
    public decimal BankFee { get; set; }
    public string BankCode { get; set; }
    
    public BankTransfer() : base() 
    {
        BankCode = "044525";
    }
    
    public BankTransfer(decimal bankFee) : base(2, "Банковский перевод", 1000m)
    {
        BankFee = bankFee;
    }
    
    public override bool CheckMinimumAmount(decimal amount)
    {
        decimal totalAmount = amount + BankFee;
        if (totalAmount >= MinAmount)
        {
            Console.WriteLine($"Средств достаточно (комиссия: {BankFee} RUB)");
            return true;
        }
        else
        {
            Console.WriteLine($"Недостаточно средств (требуется: {MinAmount} RUB)");
            return false;
        }
    }
    
    public void ProcessInternationalTransfer(decimal amount, string targetCountry)
    {
        Console.WriteLine($"Международный перевод в {targetCountry}");
        ProcessPayment(amount);
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Комиссия: {BankFee} RUB, БИК: {BankCode}");
    }
}

class CashPayment : PaymentMethod
{
    public string CashPickupPoint { get; set; }
    public string LocationAddress { get; set; }
    
    public CashPayment() : base() 
    {
        LocationAddress = "Не указан";
    }
    
    public CashPayment(string cashPickupPoint) : base(3, "Наличные", 150m)
    {
        CashPickupPoint = cashPickupPoint;
    }
    
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Обработка наличных в точке: {CashPickupPoint}");
        base.ProcessPayment(amount);
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Место выдачи: {CashPickupPoint}, Адрес: {LocationAddress}");
    }
}

class CreditCardPayment : PaymentMethod, ICurrencyConverter
{
    public string CardNumber { get; set; }
    public string CardHolder { get; set; }
    public string DefaultCurrency { get; set; }
    public string CardType { get; set; }
    
    public CreditCardPayment() : base() 
    {
        DefaultCurrency = "RUB";
        CardType = "Visa";
    }
    
    public CreditCardPayment(string cardNumber, string cardHolder) : base(4, "Кредитная карта", 50m)
    {
        CardNumber = cardNumber;
        CardHolder = cardHolder;
    }
    
    public override void ProcessPayment(decimal amount)
    {
        Console.WriteLine($"Обработка {CardType} карты");
        base.ProcessPayment(amount);
        Console.WriteLine($"Карта: ****{CardNumber.Substring(CardNumber.Length - 4)}");
    }
    
    public override void GetPaymentDetails()
    {
        base.GetPaymentDetails();
        Console.WriteLine($"Держатель: {CardHolder}, Тип: {CardType}");
    }
    
    public decimal ConvertAmount(decimal amount, string targetCurrency)
    {
        decimal convertedAmount = amount * 0.88M;
        Console.WriteLine($"Конвертация по карте: {amount} {Currency} → {convertedAmount} {targetCurrency}");
        return convertedAmount;
    }
}

Console.WriteLine("=== ДЕМОНСТРАЦИЯ СИСТЕМЫ ПЛАТЕЖЕЙ ===\n");

var onlinePayment = new OnlinePayment("https://payment-gateway.com/process")
{
    MinAmount = 100,
    GatewayProvider = "CloudPayments"
};

var bankTransfer = new BankTransfer(120)
{
    MinAmount = 1000
};

var cashPayment = new CashPayment("Сбербанк на Ленина")
{
    MinAmount = 150
};

var creditCard = new CreditCardPayment("4111111111111111", "Иванов И.И.")
{
    MinAmount = 50,
    CardType = "MasterCard"
};

Console.WriteLine("=== ПОЛИМОРФИЗМ ===");
onlinePayment.ProcessPayment(200);
onlinePayment.ProcessPayment(300, "REF12345");
bankTransfer.ProcessInternationalTransfer(1500, "США");

Console.WriteLine("\n=== ДЕТАЛИ МЕТОДОВ ===");
onlinePayment.GetPaymentDetails();
bankTransfer.GetPaymentDetails();
cashPayment.GetPaymentDetails();
creditCard.GetPaymentDetails();

Console.WriteLine("\n=== GENERIC КОЛЛЕКЦИИ ===");
var onlineCollection = new PaymentCollection<OnlinePayment>();
onlineCollection.AddPayment(onlinePayment);
onlineCollection.AddPayment(new OnlinePayment("https://backup.com") { MethodName = "Резервный" });
onlineCollection.ProcessAll(500);

Console.WriteLine("\n=== КОНВЕРТАЦИЯ ВАЛЮТ ===");
onlinePayment.ConvertAmount(1000, "USD");
creditCard.ConvertAmount(1000, "EUR");

Console.WriteLine("\n=== ДОПОЛНИТЕЛЬНЫЕ ВОЗМОЖНОСТИ ===");
onlinePayment.UpdatePriority(5);
Console.WriteLine(onlinePayment.GetStatus());
Console.WriteLine(creditCard.GetStatus());

Console.WriteLine("\n=== ИСТОРИЯ ТРАНЗАКЦИЙ ===");
Console.WriteLine(onlinePayment.GetTransactionHistory());

=== ДЕМОНСТРАЦИЯ СИСТЕМЫ ПЛАТЕЖЕЙ ===

=== ПОЛИМОРФИЗМ ===
Онлайн платеж через CloudPayments
Транзакция через Онлайн: 200 RUB
Транзакция успешна
URL: https://payment-gateway.com/process
Транзакция с референсом: REF12345
Онлайн платеж через CloudPayments
Транзакция через Онлайн: 300 RUB
Транзакция успешна
URL: https://payment-gateway.com/process
Международный перевод в США
Транзакция через Банковский перевод: 1500 RUB
Транзакция успешна

=== ДЕТАЛИ МЕТОДОВ ===
ID: 1, Способ: Онлайн, Мин. сумма: 100 RUB
Провайдер: CloudPayments, URL: https://payment-gateway.com/process
ID: 2, Способ: Банковский перевод, Мин. сумма: 1000 RUB
Комиссия: 120 RUB, БИК: 
ID: 3, Способ: Наличные, Мин. сумма: 150 RUB
Место выдачи: Сбербанк на Ленина, Адрес: 
ID: 4, Способ: Кредитная карта, Мин. сумма: 50 RUB
Держатель: Иванов И.И., Тип: MasterCard

=== GENERIC КОЛЛЕКЦИИ ===
Добавлен: Онлайн
Добавлен: Резервный
Обработка коллекции (2 элементов):
Средств достаточно
Онлайн платеж через CloudPayments
Транзакция чере